# Preamble

In [1]:
# Copy the wheel file from your bucket to the cluster's local storage
!gsutil cp gs://mksyunz-sparkly-bucket/spark-sql/sparksql_magic*.whl /tmp/

# Install the package directly from the local file
%pip install /tmp/sparksql_magic*.whl

Copying gs://mksyunz-sparkly-bucket/spark-sql/sparksql_magic-0.0.3-py36-none-any.whl...
/ [1 files][  4.2 KiB/  4.2 KiB]                                                
Operation completed over 1 objects/4.2 KiB.                                      
Processing /tmp/sparksql_magic-0.0.3-py36-none-any.whl
sparksql-magic is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext sparksql_magic

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType

# On Dataproc, this connects to the existing cluster manager
spark = SparkSession.builder.getOrCreate()
SPANNER_OPTS = {
    "projectId":  "improvingvancouver",
    "instanceId": "mksyunz-spark-dev",
    "databaseId": "demo",
}

# This grabs the underlying JVM logger to silence it completely
log4jLogger = spark._jvm.org.apache.log4j
log = log4jLogger.LogManager.getLogger("org")
log.setLevel(log4jLogger.Level.ERROR)

spark.sparkContext.setLogLevel("ERROR")

## Register the Spanner catalog 

In [4]:
spark.conf.set("spark.sql.catalog.spanner",
               "com.google.cloud.spark.spanner.SpannerCatalog")
spark.conf.set("spark.sql.catalog.spanner.projectId",  SPANNER_OPTS["projectId"])
spark.conf.set("spark.sql.catalog.spanner.instanceId", SPANNER_OPTS["instanceId"])
spark.conf.set("spark.sql.catalog.spanner.databaseId", SPANNER_OPTS["databaseId"])


print(f"Spanner: {SPANNER_OPTS['projectId']}/{SPANNER_OPTS['instanceId']}/{SPANNER_OPTS['databaseId']}")
print("✓ Spanner catalog registered.")

Spanner: improvingvancouver/mksyunz-spark-dev/demo
✓ Spanner catalog registered.


## Part 1. Create tables

In [5]:
%%sparksql
DROP TABLE IF EXISTS spanner.demo_df_write

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used


In [6]:
%%sparksql
CREATE TABLE spanner.demo_df_write (
    id     BIGINT NOT NULL,
    name   STRING,
    dept   STRING,
    salary DOUBLE
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

In [7]:
%%sparksql
DROP TABLE IF EXISTS spanner.demo_overwrite

In [8]:
%%sparksql
CREATE TABLE spanner.demo_overwrite (
    id   BIGINT NOT NULL,
    name STRING
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

# Spark Spanner Connector — Write Demo (Dataproc)

This notebook demonstrates the Spark connector for Spanner write capabilities.

It covers:
- **Part 1** — DataFrame API writes (append, overwrite, mutation types)
- **Part 2** — Spark Catalog (CREATE TABLE, INSERT INTO, SELECT, DROP TABLE)

---
# Part 1 — DataFrame API Writes

## 1a. Append mode

The default — inserts rows into the existing table.

In [9]:
schema = StructType([
    StructField("id",     LongType(),   False),
    StructField("name",   StringType(), True),
    StructField("dept",   StringType(), True),
    StructField("salary", DoubleType(), True),
])

data = [
    (1, "Alice",   "Engineering", 120000.0),
    (2, "Bob",     "Marketing",    95000.0),
    (3, "Charlie", "Engineering", 130000.0),
]
df = spark.createDataFrame(data, schema)

(df.write.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_df_write")
    .mode("append")
    .save())

print("✓ Wrote 3 rows with append mode.")

✓ Wrote 3 rows with append mode.


Saved data can now be read back:

In [10]:
%%sparksql
SELECT * FROM spanner.demo_df_write ORDER BY id

id,name,dept,salary
1,Alice,Engineering,120000.0
2,Bob,Marketing,95000.0
3,Charlie,Engineering,130000.0


## 1b. mutationType -- INSERT

The `mutationType` option controls the Spanner mutation used:
- `insert` — fails if the row already exists
- `update` — fails if the row does not exist
- `insert_or_update` (default) — upserts
- `replace` — replaces the entire row

In [11]:
new_row = spark.createDataFrame([(4, "Diana", "Sales", 105000.0)], schema)

(new_row.write.format("cloud-spanner")
    .options(**{**SPANNER_OPTS, "table": "demo_df_write", "mutationType": "insert"})
    .mode("append")
    .save())

print("✓ Inserted row id=4 with mutationType='insert'.")

✓ Inserted row id=4 with mutationType='insert'.


In [12]:
%%sparksql
SELECT * FROM spanner.demo_df_write ORDER BY id

id,name,dept,salary
1,Alice,Engineering,120000.0
2,Bob,Marketing,95000.0
3,Charlie,Engineering,130000.0
4,Diana,Sales,105000.0


Now, if row with same `id` is inserted, the save will fail:

In [13]:
new_row_2 = spark.createDataFrame([(4, "Bob", "Builder", 4445000.0)], schema)
try:
    (new_row.write.format("cloud-spanner")
        .options(**{**SPANNER_OPTS, "table": "demo_df_write", "mutationType": "insert"})
        .mode("append")
        .save())
    print("✓ Inserted row id=4 with mutationType='insert'.")
except Exception as e:
    print(f"✓ Caught exception saving with existing key using insert mutationType.")


26/03/14 01:18:34 WARN TaskSetManager: Lost task 3.0 in stage 4.0 (TID 13) (mksyunz-jupyter-b-w-1.c.improvingvancouver.internal executor 1): java.io.IOException: Failed to commit Spanner partition 3
	at com.google.cloud.spark.spanner.SpannerDataWriter.commit(SpannerDataWriter.java:183)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask.$anonfun$run$5(WriteToDataSourceV2Exec.scala:493)
	at org.apache.spark.util.Utils$.tryWithSafeFinallyAndFailureCallbacks(Utils.scala:1399)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask.run(WriteToDataSourceV2Exec.scala:523)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask.run$(WriteToDataSourceV2Exec.scala:459)
	at org.apache.spark.sql.execution.datasources.v2.DataWritingSparkTask$.run(WriteToDataSourceV2Exec.scala:528)
	at org.apache.spark.sql.execution.datasources.v2.V2TableWriteExec.$anonfun$writeWithV2$2(WriteToDataSourceV2Exec.scala:408)
	at org.apache.spark.scheduler.ResultTask.runTask(ResultTa

✓ Caught exception saving with existing key using insert mutationType.


26/03/14 01:18:35 ERROR TaskSetManager: Task 3 in stage 4.0 failed 4 times; aborting job
26/03/14 01:18:35 ERROR AppendDataExec: Data source write support com.google.cloud.spark.spanner.SpannerBatchWrite@2acc976f is aborting.
26/03/14 01:18:35 ERROR AppendDataExec: Data source write support com.google.cloud.spark.spanner.SpannerBatchWrite@2acc976f aborted.


## 1c. Overwrite mode (truncate)

`mode("overwrite")` with `overwriteMode=truncate` deletes all existing rows, then writes the new data. The table schema is preserved.

First, create an index on table `demo_overwrite`:

In [14]:
from google.cloud import spanner

# Initialize the client
spanner_client = spanner.Client(project=SPANNER_OPTS["projectId"])
instance = spanner_client.instance(SPANNER_OPTS["instanceId"])
database = instance.database(SPANNER_OPTS["databaseId"])

# Define the DDL command
index_name = "demo_overwrite_name_idx"
ddl_statement = f"CREATE INDEX {index_name} ON demo_overwrite(name)"

print(f"Creating index: {index_name}...")

# Submit the operation and wait for it to finish
operation = database.update_ddl([ddl_statement])
operation.result(timeout=300) # Waits up to 5 minutes for Spanner to backfill the index

print("Index creation successfully finished!")

Creating index: demo_overwrite_name_idx...
Index creation successfully finished!


In [15]:
small_schema = StructType([
    StructField("id",   LongType(),   False, {"spanner.primaryKey": True}),
    StructField("name", StringType(), True),
])

initial = spark.createDataFrame([(1, "old-row-1"), (2, "old-row-2")], small_schema)
(initial.write.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_overwrite")
    .mode("overwrite")
    .save())

print("✓ Seeded demo_overwrite with 2 rows.")

✓ Seeded demo_overwrite with 2 rows.


In [16]:
%%sparksql
SELECT * FROM spanner.demo_overwrite ORDER BY id

id,name
1,old-row-1
2,old-row-2


In [17]:
replacement = spark.createDataFrame([(10, "new-row-10"), (20, "new-row-20")], small_schema)

(replacement.write.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_overwrite")
    .option("overwriteMode", "truncate")
    .mode("overwrite")
    .save())

print("✓ Overwrote with 2 new rows (truncate mode).")

✓ Overwrote with 2 new rows (truncate mode).


In [18]:
%%sparksql
SELECT * FROM spanner.demo_overwrite ORDER BY id

id,name
10,new-row-10
20,new-row-20


## 1d. Overwrite mode (recreate)
`mode("overwrite")` with `overwriteMode=recreate` deletes the existing table, creates a new one with the same name, then writes the new data.

It is much faster but table schema is not preserved.

If we try to use it now, it will fail because index existence prevents table deletion.

In [19]:
from py4j.protocol import Py4JJavaError

replacement2 = spark.createDataFrame([(10, "recreate-new-row-10"), (20, "recreate-new-row-20")], small_schema)

try:
    (replacement2.write.format("cloud-spanner")
        .options(**SPANNER_OPTS)
        .option("table", "demo_overwrite")
        .option("overwriteMode", "recreate")
        .mode("overwrite")
        .save())
except Py4JJavaError as e:
    print(f"✓ Recreate failed because index exists: {e.java_exception.getMessage()}")

✓ Recreate failed because index exists: Error recreating table demo_overwrite


However, if we remove the index:

In [20]:
ddl_statement = f"DROP INDEX {index_name}"

print(f"Dropping index: {index_name}...")

# Submit the operation and wait for it to finish
operation = database.update_ddl([ddl_statement])
operation.result(timeout=300)

print("Index successfully dropped!")

Dropping index: demo_overwrite_name_idx...
Index successfully dropped!


we will can recreate the table:

In [21]:
(replacement2.write.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_overwrite")
    .option("overwriteMode", "recreate")
    .mode("overwrite")
    .save())

and check new values:

In [22]:
%%sparksql
SELECT * FROM spanner.demo_overwrite ORDER BY id

id,name
10,recreate-new-row-10
20,recreate-new-row-20


---
# Part 2 — Spark Catalog (SQL)

- Simplifies working with Spanner databases.
- Alows managing tables from Spark.

In [23]:
%%sparksql
DROP TABLE IF EXISTS spanner.demo_catalog

In [24]:
%%sparksql
CREATE TABLE spanner.demo_catalog (
    id     BIGINT NOT NULL,
    city   STRING,
    temp_f DOUBLE
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

## 2b. INSERT INTO

In [25]:
%%sparksql
INSERT INTO spanner.demo_catalog VALUES
    (1, 'San Francisco', 62.0),
    (2, 'New York', 45.0),
    (3, 'Austin', 85.0)

## 2c. SELECT

In [26]:
%%sparksql

SELECT * FROM spanner.demo_catalog ORDER BY id

id,city,temp_f
1,San Francisco,62.0
2,New York,45.0
3,Austin,85.0


## 2d. writeTo().create() — ErrorIfExists Save Mode

In [27]:
%%sparksql
DROP TABLE IF EXISTS spanner.demo_catalog_eie

In [28]:
df = spark.createDataFrame(
    [(100, "Seattle", 55.0)],
    StructType([
        StructField("id",     LongType(),   False),
        StructField("city",   StringType(), True),
        StructField("temp_f", DoubleType(), True),
    ]),
)

df.writeTo("spanner.demo_catalog_eie").tableProperty("primaryKeys", "id").create()
print("✓ Created demo_catalog_eie and wrote 1 row.")

✓ Created demo_catalog_eie and wrote 1 row.


In [29]:
%%sparksql
SELECT * FROM spanner.demo_catalog_eie ORDER BY id

id,city,temp_f
100,Seattle,55.0


In [30]:
# Second create should fail (table already exists)
try:
    df.writeTo("spanner.demo_catalog_eie").tableProperty("primaryKeys", "id").create()
    print("✗ Expected error was NOT raised!")
except Exception as e:
    print(f"✓ Second create() correctly failed: {e}")

✓ Second create() correctly failed: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `demo_catalog_eie` because it already exists.
Choose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects.


## 2e. CREATE TABLE IF NOT EXISTS — Ignore Save Mode

In [31]:
%%sparksql
DROP TABLE IF EXISTS spanner.demo_catalog_ign

Create table `demo_catalog_ign`:

In [32]:
%%sparksql
CREATE TABLE IF NOT EXISTS spanner.demo_catalog_ign (
    id   BIGINT NOT NULL,
    note STRING
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

Insert some data

In [33]:
%%sparksql
INSERT INTO spanner.demo_catalog_ign VALUES (1, 'first')

Try to create `demo-catalog_ign` table again with `IF NOT EXISTS` guard:

In [34]:
%%sparksql
-- Second CREATE TABLE IF NOT EXISTS is a no-op
CREATE TABLE IF NOT EXISTS spanner.demo_catalog_ign (
    id   BIGINT NOT NULL,
    note STRING
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

It was a no-op.

Let's confirm that table still has data:

In [35]:
%%sparksql
SELECT * FROM spanner.demo_catalog_ign ORDER BY id

id,note
1,first


## 3a. Create a table with a JSON column

In [36]:
%%sparksql
DROP TABLE IF EXISTS spanner.demo_json

In [37]:
%%sparksql
CREATE TABLE spanner.demo_json (
    id       BIGINT NOT NULL,
    name     STRING,
    metadata STRING
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

## 3b. Build a DataFrame with a struct column and convert to JSON

In [38]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, ArrayType

# Define a schema with a nested struct column
nested_schema = StructType([
    StructField("id",   LongType(), False),
    StructField("name", StringType(), True),
    StructField("metadata", StructType([
        StructField("role",       StringType(), True),
        StructField("level",      IntegerType(), True),
        StructField("skills",     ArrayType(StringType()), True),
    ]), True),
])

data = [
    (1, "Alice",   ("engineer",  5, ["python", "spark", "sql"])),
    (2, "Bob",     ("analyst",   3, ["sql", "tableau"])),
    (3, "Charlie", ("manager",   7, ["leadership", "strategy"])),
]
struct_df = spark.createDataFrame(data, nested_schema)

print("Original DataFrame with struct column:")
struct_df.printSchema()
struct_df.show(truncate=False)

Original DataFrame with struct column:
root
 |-- id: long (nullable = false)
 |-- name: string (nullable = true)
 |-- metadata: struct (nullable = true)
 |    |-- role: string (nullable = true)
 |    |-- level: integer (nullable = true)
 |    |-- skills: array (nullable = true)
 |    |    |-- element: string (containsNull = true)

+---+-------+------------------------------------+
|id |name   |metadata                            |
+---+-------+------------------------------------+
|1  |Alice  |{engineer, 5, [python, spark, sql]} |
|2  |Bob    |{analyst, 3, [sql, tableau]}        |
|3  |Charlie|{manager, 7, [leadership, strategy]}|
+---+-------+------------------------------------+



In [39]:
# Convert the struct column to a JSON string for writing to Spanner
write_df = struct_df.withColumn("metadata", F.to_json("metadata"))

print("DataFrame after to_json() — ready to write to Spanner:")
write_df.printSchema()
write_df.show(truncate=False)

DataFrame after to_json() — ready to write to Spanner:
root
 |-- id: long (nullable = false)
 |-- name: string (nullable = true)
 |-- metadata: string (nullable = true)

+---+-------+---------------------------------------------------------------+
|id |name   |metadata                                                       |
+---+-------+---------------------------------------------------------------+
|1  |Alice  |{"role":"engineer","level":5,"skills":["python","spark","sql"]}|
|2  |Bob    |{"role":"analyst","level":3,"skills":["sql","tableau"]}        |
|3  |Charlie|{"role":"manager","level":7,"skills":["leadership","strategy"]}|
+---+-------+---------------------------------------------------------------+



## 3c. Write to Spanner

In [40]:
(write_df.write.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_json")
    .mode("append")
    .save())

print("✓ Wrote 3 rows with JSON metadata to Spanner.")

✓ Wrote 3 rows with JSON metadata to Spanner.


---
## Cleanup

In [44]:
%%sparksql
DROP TABLE IF EXISTS spanner.demo_df_write

In [45]:
%%sparksql
DROP TABLE IF EXISTS spanner.demo_catalog

In [46]:
%%sparksql
DROP TABLE IF EXISTS spanner.demo_overwrite

In [47]:
%%sparksql

DROP TABLE IF EXISTS spanner.demo_catalog_eie

In [48]:
%%sparksql
DROP TABLE IF EXISTS spanner.demo_catalog_ign